In [10]:
from pathlib import Path

# 建议从 Notebook 所在目录启动 Jupyter/VS Code。
# 如果当前工作目录不同，可以手工把这里改成 knowledge_base 的绝对路径。
KNOWLEDGE_DIR = Path.cwd() / "knowledge_base"

if not KNOWLEDGE_DIR.exists():
    raise FileNotFoundError(
        f"没有找到知识库目录：{KNOWLEDGE_DIR}\n"
        "请先将当前工作目录切换到 Notebook 所在文件夹，"
        "或者手工修改 KNOWLEDGE_DIR。"
    )

docx_files = sorted(KNOWLEDGE_DIR.glob("*.docx"))
if not docx_files:
    raise FileNotFoundError(f"{KNOWLEDGE_DIR} 中没有 .docx 文件")

print(f"知识库目录：{KNOWLEDGE_DIR}")
print(f"发现 {len(docx_files)} 份 Word 文档：")
for path in docx_files:
    print("-", path.name)


知识库目录：/Users/jackhu/src_code/ai-infra-agent/agent-tutorial/ragas-demo/knowledge_base
发现 6 份 Word 文档：
- 01_员工休假制度.docx
- 02_费用报销制度.docx
- 03_远程办公制度.docx
- 04_办公区域安全制度.docx
- 05_线上故障响应制度.docx
- 06_员工培训制度.docx


In [11]:
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.embeddings.dashscope import (
    DashScopeEmbedding,
    DashScopeTextEmbeddingModels,
    DashScopeTextEmbeddingType,
)
from llama_index.llms.dashscope import DashScope
from llama_index.readers.file import DocxReader
import os

embed_model = DashScopeEmbedding(
    model_name=DashScopeTextEmbeddingModels.TEXT_EMBEDDING_V3,
    text_type=DashScopeTextEmbeddingType.TEXT_TYPE_DOCUMENT,
)

llm = DashScope(
    model_name="glm-5.2",
    api_key=os.environ["DASHSCOPE_API_KEY"],
    temperature=0.0,
)

Settings.embed_model = embed_model
Settings.llm = llm

print("Embedding 模型和 LLM 配置完成")


Embedding 模型和 LLM 配置完成


## 3. 构建 LlamaIndex 向量索引

`SimpleDirectoryReader` 负责扫描目录，`DocxReader` 负责从 Word 文件中提取文本。加载完成后，每个 LlamaIndex Document 都会带有 `file_name`、`file_path` 等 Metadata。后续 Node 会继承这些信息，因此系统能够追踪召回内容来自哪份文件，而不需要修改原始 Word 文档。


In [13]:
documents = SimpleDirectoryReader(
    input_dir=str(KNOWLEDGE_DIR),
    required_exts=[".docx"],
    file_extractor={".docx": DocxReader()},
    recursive=False,
    raise_on_error=True,
).load_data()

print(f"LlamaIndex 实际加载了 {len(documents)} 个 Document")
for document in documents:
    print(
        document.metadata.get("file_name"),
        "| 字符数：",
        len(document.text),
    )

index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embed_model,
    show_progress=True,
)

retriever = index.as_retriever(similarity_top_k=2)
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=2,
    response_mode="compact",
)

print("索引构建完成，Node 数量：", len(index.docstore.docs))


LlamaIndex 实际加载了 6 个 Document
01_员工休假制度.docx | 字符数： 399
02_费用报销制度.docx | 字符数： 341
03_远程办公制度.docx | 字符数： 378
04_办公区域安全制度.docx | 字符数： 360
05_线上故障响应制度.docx | 字符数： 370
06_员工培训制度.docx | 字符数： 337


Generating embeddings: 100%|██████████| 6/6 [00:02<00:00,  2.49it/s]

索引构建完成，Node 数量： 6


In [14]:
demo_question = "员工的门禁卡丢了，应该在多长时间内报告？"
demo_response = query_engine.query(demo_question)

print("问题：", demo_question)
print("回答：", str(demo_response))
print("\n召回上下文：")

for rank, source_node in enumerate(demo_response.source_nodes, start=1):
    node = source_node.node
    source_file = node.metadata.get("file_name")
    if not source_file and node.metadata.get("file_path"):
        source_file = Path(node.metadata["file_path"]).name
    print(
        f"\nTop {rank} | 来源文件={source_file} "
        f"| score={source_node.score:.4f}"
    )
    print(node.get_content())


问题： 员工的门禁卡丢了，应该在多长时间内报告？
回答： 员工发现门禁卡遗失后，必须在30分钟内向行政部和信息安全团队报告。

召回上下文：

Top 1 | 来源文件=04_办公区域安全制度.docx | score=0.7734
示例科技有限公司｜内部制度文件

办公区域安全制度

企业内部管理制度

归口部门：行政部、信息安全团队

适用范围：进入公司办公区域的员工和长期驻场人员

生效日期：2026年1月1日

文件版本：V1.0



一、制度目的

规范办公区域身份凭证的使用和遗失处理，降低未授权访问风险。

二、制度内容

第一条 门禁卡使用

员工应妥善保管本人门禁卡，不得把门禁卡借给他人使用，也不得代替无权限人员刷卡进入受控区域。

第二条 遗失报告

员工发现门禁卡遗失后，必须在30分钟内向行政部和信息安全团队报告。报告时应说明最后使用时间和可能遗失地点。

第三条 停用与补办

行政部收到报告后会立即停用原门禁卡并安排补办。员工找回已停用的门禁卡后不得自行恢复使用，应交回行政部处理。

内部资料，请妥善保管｜第 1 页

Top 2 | 来源文件=03_远程办公制度.docx | score=0.5746
示例科技有限公司｜内部制度文件

远程办公制度

企业内部管理制度

归口部门：人力资源部、信息技术部

适用范围：通过试用期且岗位适合远程协作的员工

生效日期：2026年1月1日

文件版本：V1.0



一、制度目的

明确远程办公的适用时间、申请方式和信息安全要求，保证远程协作效率。

二、制度内容

第一条 适用时间

符合条件的员工可以在每周三申请远程办公。遇到公司级会议、客户现场工作或团队集中协作安排时，应优先服从现场办公要求。

第二条 申请要求

员工应在前一个工作日下班前提交远程办公申请，经直属主管确认后方可执行。远程办公地点应具备稳定网络和安静的工作环境。

第三条 安全与在线要求

远程办公期间必须通过公司VPN访问内部系统，并在工作时间保持即时通讯在线。不得使用公共设备保存公司资料。

内部资料，请妥善保管｜第 1 页


In [15]:
EVAL_SET = [
    {
        "question": "正式员工每年有多少带薪年假？",
        "reference_answer": "正式员工每个自然年度享有10个工作日的带薪年假。",
        "reference_keyword": "10个工作日",
        "reference_source_files": ["01_员工休假制度.docx"],
    },
    {
        "question": "费用发生后最晚多久要提交报销？",
        "reference_answer": "员工应在费用发生后的5个工作日内提交报销申请。",
        "reference_keyword": "5个工作日",
        "reference_source_files": ["02_费用报销制度.docx"],
    },
    {
        "question": "门禁卡丢失后必须在多久内报告？",
        "reference_answer": "门禁卡遗失后必须在30分钟内报告。",
        "reference_keyword": "30分钟",
        "reference_source_files": ["04_办公区域安全制度.docx"],
    },
    {
        "question": "P1级线上故障需要在多长时间内确认告警？",
        "reference_answer": "值班工程师必须在10分钟内确认P1级线上故障告警。",
        "reference_keyword": "10分钟",
        "reference_source_files": ["05_线上故障响应制度.docx"],
    },
    {
        "question": "员工可以在每周哪一天申请远程办公？",
        "reference_answer": "符合条件的员工可以在每周三申请远程办公。",
        "reference_keyword": "每周三",
        "reference_source_files": ["03_远程办公制度.docx"],
    },
]

print(f"Golden Dataset 包含 {len(EVAL_SET)} 个问题")


Golden Dataset 包含 5 个问题


In [16]:
def get_source_file_name(source_node) -> str:
    node = source_node.node
    file_name = node.metadata.get("file_name")
    if file_name:
        return str(file_name)

    file_path = node.metadata.get("file_path")
    if file_path:
        return Path(file_path).name

    # 正常的 SimpleDirectoryReader + DocxReader 不会走到这里。
    return "unknown_source"


rag_records = []

for number, case in enumerate(EVAL_SET, start=1):
    response = query_engine.query(case["question"])
    source_nodes = response.source_nodes

    record = {
        **case,
        "response": str(response),
        "retrieved_contexts": [
            source.node.get_content() for source in source_nodes
        ],
        "retrieved_source_files": [
            get_source_file_name(source) for source in source_nodes
        ],
    }
    rag_records.append(record)
    print(f"[{number}/{len(EVAL_SET)}] {case['question']}")
    print("回答：", record["response"])
    print("召回来源：", record["retrieved_source_files"])
    print()

print("批量 RAG 执行完成")


[1/5] 正式员工每年有多少带薪年假？
回答： 正式员工每个自然年度享有10个工作日的带薪年假。
召回来源： ['01_员工休假制度.docx', '06_员工培训制度.docx']

[2/5] 费用发生后最晚多久要提交报销？
回答： 费用发生后，员工应在5个工作日内提交报销申请。
召回来源： ['02_费用报销制度.docx', '06_员工培训制度.docx']

[3/5] 门禁卡丢失后必须在多久内报告？
回答： 门禁卡丢失后，必须在30分钟内向行政部和信息安全团队报告。
召回来源： ['04_办公区域安全制度.docx', '02_费用报销制度.docx']

[4/5] P1级线上故障需要在多长时间内确认告警？
回答： P1级线上故障需要在10分钟内确认告警。
召回来源： ['05_线上故障响应制度.docx', '02_费用报销制度.docx']

[5/5] 员工可以在每周哪一天申请远程办公？
回答： 员工可以在每周三申请远程办公。
召回来源： ['03_远程办公制度.docx', '01_员工休假制度.docx']

批量 RAG 执行完成


In [19]:
import pandas as pd
from ragas import SingleTurnSample
from ragas.metrics.collections import (
    NonLLMStringSimilarity,
    StringPresence,
)

try:
    from ragas.metrics.collections import (
        IDBasedContextPrecision,
        IDBasedContextRecall,
    )
except ImportError:
    from ragas.metrics._context_precision import IDBasedContextPrecision
    from ragas.metrics._context_recall import IDBasedContextRecall

context_precision_metric = IDBasedContextPrecision()
context_recall_metric = IDBasedContextRecall()
keyword_metric = StringPresence()
answer_similarity_metric = NonLLMStringSimilarity()

evaluation_rows = []

for record in rag_records:
    sample = SingleTurnSample(
        user_input=record["question"],
        response=record["response"],
        reference=record["reference_answer"],
        retrieved_contexts=record["retrieved_contexts"],
        # Ragas 的字段名叫 context_ids，但这里传入的是来源文件名。
        # 它们是加载时生成的追踪标签，不是 Word 文档自带的 ID。
        retrieved_context_ids=record["retrieved_source_files"],
        reference_context_ids=record["reference_source_files"],
    )

    id_precision = context_precision_metric.single_turn_score(sample)
    id_recall = context_recall_metric.single_turn_score(sample)
    keyword_score = (await keyword_metric.ascore(
        reference=record["reference_keyword"],
        response=record["response"],
    )).value
    answer_similarity = (await answer_similarity_metric.ascore(
        reference=record["reference_answer"],
        response=record["response"],
    )).value

    evaluation_rows.append(
        {
            "question": record["question"],
            "retrieved_files": ", ".join(record["retrieved_source_files"]),
            "response": record["response"],
            "id_context_precision": float(id_precision),
            "id_context_recall": float(id_recall),
            "keyword_presence": float(keyword_score),
            "answer_string_similarity": float(answer_similarity),
        }
    )

results_df = pd.DataFrame(evaluation_rows)
results_df.round(3)


,question,retrieved_files,response,id_context_precision,id_context_recall,keyword_presence,answer_string_similarity
0,正式员工每年有多少带薪年假？,"01_员工休假制度.docx, 06_员工培训制度.docx",正式员工每个自然年度享有10个工作日的带薪年假。,0.5,1.0,1.0,1.000
1,费用发生后最晚多久要提交报销？,"02_费用报销制度.docx, 06_员工培训制度.docx",费用发生后，员工应在5个工作日内提交报销申请。,0.5,1.0,1.0,0.609
2,门禁卡丢失后必须在多久内报告？,"04_办公区域安全制度.docx, 02_费用报销制度.docx",门禁卡丢失后，必须在30分钟内向行政部和信息安全团队报告。,0.5,1.0,1.0,0.552
3,P1级线上故障需要在多长时间内确认告警？,"05_线上故障响应制度.docx, 02_费用报销制度.docx",P1级线上故障需要在10分钟内确认告警。,0.5,1.0,1.0,0.360
4,员工可以在每周哪一天申请远程办公？,"03_远程办公制度.docx, 01_员工休假制度.docx",员工可以在每周三申请远程办公。,0.5,1.0,1.0,0.750


In [20]:
metric_columns = [
    "id_context_precision",
    "id_context_recall",
    "keyword_presence",
    "answer_string_similarity",
]

summary = results_df[metric_columns].mean().to_frame("mean_score")
display(summary.round(3))

QUALITY_GATES = {
    "id_context_recall": 0.90,
    "keyword_presence": 0.90,
}

print("质量门槛检查：")
for metric_name, threshold in QUALITY_GATES.items():
    actual = float(summary.loc[metric_name, "mean_score"])
    status = "PASS" if actual >= threshold else "FAIL"
    print(f"{status} | {metric_name}: {actual:.3f} >= {threshold:.2f}")


,mean_score
id_context_precision,0.500
id_context_recall,1.000
keyword_presence,1.000
answer_string_similarity,0.654


质量门槛检查：
PASS | id_context_recall: 1.000 >= 0.90
PASS | keyword_presence: 1.000 >= 0.90


In [21]:
failed_cases = results_df[
    (results_df["id_context_recall"] < 1.0)
    | (results_df["keyword_presence"] < 1.0)
]

if failed_cases.empty:
    print("没有发现检索遗漏或关键事实遗漏。")
else:
    print(f"发现 {len(failed_cases)} 条需要分析的案例：")
    display(
        failed_cases[
            [
                "question",
                "retrieved_files",
                "response",
                "id_context_recall",
                "keyword_presence",
            ]
        ]
    )


没有发现检索遗漏或关键事实遗漏。


## 10. 可选：使用 Ragas 的 LLM Judge 指标

上面的主评估已经完整使用 Ragas，且不额外消耗 Judge 模型 Token。若希望评估 Faithfulness，可以把下面的 `RUN_LLM_JUDGE` 改为 `True`。

该代码通过 DashScope 的 OpenAI-compatible endpoint 创建 Ragas Judge。Judge 本身也可能产生误判，因此企业项目中应先抽样进行人工校准，再决定阈值。


In [24]:
RUN_LLM_JUDGE = True

if RUN_LLM_JUDGE:
    from openai import AsyncOpenAI
    from ragas.llms import llm_factory
    from ragas.metrics.collections import Faithfulness

    judge_client = AsyncOpenAI(
        api_key=os.environ["DASHSCOPE_API_KEY"],
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    )
    judge_llm = llm_factory(
        model="glm-5.2",
        provider="openai",
        client=judge_client,
    )
    faithfulness_metric = Faithfulness(llm=judge_llm)

    faithfulness_rows = []
    for record in rag_records:
        result = await faithfulness_metric.ascore(
            user_input=record["question"],
            response=record["response"],
            retrieved_contexts=record["retrieved_contexts"],
        )
        faithfulness_rows.append(
            {
                "question": record["question"],
                "faithfulness": result.value,
                "reason": result.reason,
            }
        )

    display(pd.DataFrame(faithfulness_rows))
else:
    print("已跳过可选 LLM Judge；主 Ragas 评估不受影响。")


2026-08-29 10:06:18,802 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-29 10:06:28,709 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-29 10:06:35,494 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-29 10:06:41,448 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-29 10:06:52,227 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-29 10:07:08,092 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-29 10:07:21,405 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-29 10:07:28,650 - INFO - HTTP Req

,question,faithfulness,reason
0,正式员工每年有多少带薪年假？,1.0,None
1,费用发生后最晚多久要提交报销？,1.0,None
2,门禁卡丢失后必须在多久内报告？,1.0,None
3,P1级线上故障需要在多长时间内确认告警？,1.0,None
4,员工可以在每周哪一天申请远程办公？,1.0,None
